# The Price Is Right - Week 7

## Day 3 and Day 4: Training!

This is what it's all been about!

If you are using LITE_MODE=True, then please run this on a free T4 box.

If you are using LITE_MODE-False, then please use a paid A100 with high memory.

In [ ]:
!wget -q https://github.com/Abhishekravindran/LLM_Engineering_opensource/tree/main/finetuning_local_frontier_models/util.py -O util.py

In [ ]:
# ============================================================
# Clean installation for FP16 LoRA fine-tuning
# Google Colab
# ============================================================

!pip uninstall -y -q \
    transformers \
    huggingface_hub \
    datasets \
    peft \
    trl \
    bitsandbytes \
    accelerate \
    wandb

!pip install -U -q \
    transformers \
    huggingface_hub \
    datasets \
    peft \
    trl \
    accelerate \
    wandb

# Optional utilities
!pip install -U -q tqdm matplotlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.6/29.6 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-semantic-conventions 0.63b1 requires opentelemetry-api==1.42.1, but you have opentelemetry-

In [ ]:
!pip install -U bitsandbytes transformers accelerate peft trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 25.6 MB/s eta 0:00:00


In [ ]:
import os
import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed, BitsAndBytesConfig
from datasets import load_dataset, Dataset, DatasetDict
import wandb
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datetime import datetime
import matplotlib.pyplot as plt

In [ ]:
# Constants

BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "price"
HF_USER = "RahulHFAC" # your HF name here!

LITE_MODE = True

DATA_USER = "RahulHFAC"
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
if LITE_MODE:
  RUN_NAME += "-lite"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# Hyper-parameters - overall

EPOCHS = 1 if LITE_MODE else 3
BATCH_SIZE = 32 if LITE_MODE else 256
MAX_SEQUENCE_LENGTH = 128
GRADIENT_ACCUMULATION_STEPS = 1

# Hyper-parameters - QLoRA

QUANT_4_BIT = True
LORA_R = 32 if LITE_MODE else 256
LORA_ALPHA = LORA_R * 2
ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
MLP_LAYERS = ["gate_proj", "up_proj", "down_proj"]
TARGET_MODULES = ATTENTION_LAYERS if LITE_MODE else ATTENTION_LAYERS + MLP_LAYERS
LORA_DROPOUT = 0.1

# Hyper-parameters - training

LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.01
LR_SCHEDULER_TYPE = 'cosine'
WEIGHT_DECAY = 0.001
OPTIMIZER = "paged_adamw_32bit"

capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

# Tracking

VAL_SIZE = 500 if LITE_MODE else 1000
LOG_STEPS = 5 if LITE_MODE else 10
SAVE_STEPS = 100 if LITE_MODE else 200
LOG_TO_WANDB = True

In [ ]:
# A100 GPU supports this; T4 does not natively

use_bf16

False

# More on Optimizers

https://huggingface.co/docs/transformers/main/en/perf_train_gpu_one#optimizers

The most common is Adam or AdamW (Adam with Weight Decay).  
Adam achieves good convergence by storing the rolling average of the previous gradients; however, it adds an additional memory footprint of the order of the number of model parameters.


### Log in to HuggingFace and Weights & Biases

If you don't already have a HuggingFace account, visit https://huggingface.co to sign up and create a token.

Then select the Secrets for this Notebook by clicking on the key icon in the left, and add a new secret called `HF_TOKEN` with the value as your token.

Repeat this for weightsandbiases at https://wandb.ai and add a secret called `WANDB_API_KEY`

In [ ]:
# Log in to HuggingFace

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
# Log in to Weights & Biases
wandb_api_key = userdata.get('WANDB_API_KEY')
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

# Configure Weights & Biases to record against our project
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: rahulbhargavrgk (rahulbhargavrgk-v-align-technologies-pvt-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
dataset = load_dataset(DATASET_NAME)
train = dataset['train']
val = dataset['val'].select(range(VAL_SIZE))
test = dataset['test']

In [ ]:
# if you wish to reduce the training dataset to 10,000 points instead, then uncomment this line:

train = train.select(range(10000))

In [ ]:
if LOG_TO_WANDB:
  wandb.init(project=PROJECT_NAME, name=RUN_NAME)

## Now load the Tokenizer and Model

The model is "quantized" - we are reducing the precision to 4 bits.

In [ ]:
# pick the right quantization

if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
  )

In [ ]:
from transformers.utils import import_utils

print("Before clearing:")
print(import_utils.is_bitsandbytes_available())

import_utils.is_bitsandbytes_available.cache_clear()

print("After clearing:")
print(import_utils.is_bitsandbytes_available())

Before clearing:
True
After clearing:
True


In [ ]:
# Load the Tokenizer and the Model

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    dtype=torch.float16,
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Memory footprint: 2197.6 MB


In [ ]:
# to patch the problem creating code run if and only if you face problem related to bf16 datatype
import inspect
from trl import SFTTrainer

trl_file = inspect.getsourcefile(SFTTrainer)
print(trl_file)
print(inspect.getsourcefile(SFTTrainer.__init__))

/usr/local/lib/python3.13/dist-packages/trl/trainer/sft_trainer.py
/usr/local/lib/python3.13/dist-packages/trl/trainer/sft_trainer.py


In [ ]:
# to creates the cop of the sft_trainer code to retainback fi something goes wrong
import shutil
#create a backup
trl_file = "/usr/local/lib/python3.13/dist-packages/trl/trainer/sft_trainer.py"
backup_file = trl_file + ".backup"

shutil.copy2(trl_file, backup_file)

print("Backup created:", backup_file)

Backup created: /usr/local/lib/python3.13/dist-packages/trl/trainer/sft_trainer.py.backup


In [ ]:
# check if the code that is creatig problem is present in code
with open(trl_file, "r") as f:
    source = f.read()

old_block = """        if _is_quantized_model:
            for param in model.parameters():
                if param.requires_grad:
                    param.data = param.data.to(torch.bfloat16)
"""

print("Block found:", old_block in source)

Block found: False


In [ ]:
# run this if and only if the previous code gives true
# this code will replace the error prone code to code that gives proper op
# convert to bfloat only when you provide the arg as true and if it is a qunantized model
new_block = """        if _is_quantized_model and args.bf16:
            for param in model.parameters():
                if param.requires_grad:
                    param.data = param.data.to(torch.bfloat16)
"""

source = source.replace(old_block, new_block, 1)

with open(trl_file, "w") as f:
    f.write(source)

print("TRL patch applied.")

TRL patch applied.


# AND NOW

## We set up the configuration for Training

We need to create 2 objects:

A LoraConfig object with our hyperparameters for LoRA

An SFTConfig with our overall Training parameters

In [ ]:
# LoRA Parameters

lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

In [ ]:
train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,

    num_train_epochs=EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    optim=OPTIMIZER,

    save_steps=SAVE_STEPS,
    save_total_limit=10,

    logging_steps=LOG_STEPS,

    learning_rate=LEARNING_RATE,
    weight_decay=0.001,

    fp16=not use_bf16,
    bf16=use_bf16,

    max_grad_norm=0.3,
    max_steps=-1,

    # warmup_steps=???,

    lr_scheduler_type=LR_SCHEDULER_TYPE,

    report_to="wandb" if LOG_TO_WANDB else None,
    run_name=RUN_NAME,

    max_length=MAX_SEQUENCE_LENGTH,

    save_strategy="steps",

    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,

    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
)

In [ ]:
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    eval_dataset=val,
    peft_config=lora_parameters,
    args=train_parameters,
)

Adding EOS to train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
from collections import Counter

print("Trainer FP16:", fine_tuning.args.fp16)
print("Trainer BF16:", fine_tuning.args.bf16)

print(
    "Trainable dtypes:",
    Counter(
        str(p.dtype)
        for p in fine_tuning.model.parameters()
        if p.requires_grad
    )
)

Trainer FP16: True
Trainer BF16: False
Trainable dtypes: Counter({'torch.float32': 224})


In [ ]:
adapter = fine_tuning.model.base_model.model.model.layers[0].self_attn.q_proj

print("LoRA A:", adapter.lora_A["default"].weight.dtype)
print("LoRA B:", adapter.lora_B["default"].weight.dtype)

LoRA A: torch.float32
LoRA B: torch.float32


# AND NOW - create the trainer

## In the next cell, we kick off fine-tuning!

This will run for some time, uploading to the hub every SAVE_STEPS steps.

After some time, Google might stop your colab. For people on free plans, it can happen whenever Google is low on resources. For anyone on paid plans, they can give you up to 24 hours, but there's no guarantee.

If your server is stopped, you can follow my colab here to resume from your last save:

https://colab.research.google.com/drive/1dO20bZHxjpswj3CKysMaZku6PDgEG0SA

I've saved this colab with my final run in the output so you can see the example. The trick is that I needed to set `is_trainable=True` when loading the fine_tuned model.

### Anyway, with that in mind, let's kick this off!

In [19]:
# Fine-tune!
fine_tuning.train()

# Push our fine-tuned model to Hugging Face
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Saved to the hub: {PROJECT_RUN_NAME}")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,1.297725,1.283771,1.255118,328543.000000,0.759000
200,1.254159,1.262926,1.300592,657400.000000,0.763500
300,1.255556,1.257080,1.259581,988004.000000,0.764500
313,1.293571,1.256852,1.259289,1029278.000000,0.764500


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 16.7kB / 73.4MB            

Saved to the hub: price-2026-08-25_08.37.48-lite


In [20]:
if LOG_TO_WANDB:
  wandb.finish()

eval/entropy,▁█▂▂
eval/loss,█▃▁▁
eval/mean_token_accuracy,▁▇██
eval/num_tokens,▁▄██
eval/runtime,▄█▁▆
eval/samples_per_second,▄▁█▃
eval/steps_per_second,▄▁█▃
train/entropy,█▃▂▄▃▁▂▂▃▃▂▂▂▃▁▄▁▃▃▂▃▃▂▂▂▃▂▂▂▂▂▂▁▂▁▂▂▂▂▂
train/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train/global_step,▁▁▁▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
+5,...
